# HLTV scrape

In [ ]:
# Rankings

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from datetime import datetime

def scrape_team_rankings():
    driver = webdriver.Chrome()
    driver.get("https://www.hltv.org/ranking/teams/")
    
    # Várj, amíg betölt a lista
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ranked-team.standard-box")))
    
    rankings = []
    rank_divs = driver.find_elements(By.CSS_SELECTOR, ".ranked-team.standard-box")
    
    for rank_div in rank_divs:
        try:
            rank = rank_div.find_element(By.CLASS_NAME, "position").text
            team_name = rank_div.find_element(By.CLASS_NAME, "name").text
            points = rank_div.find_element(By.CLASS_NAME, "points").text.replace('(', '').replace(')', '').replace(" HLTV points", "")
            
            team_link = rank_div.find_element(By.TAG_NAME, "a").get_attribute("href")
            team_id = team_link.split('/')[-2]
            profile_link = rank_div.find_element(By.CLASS_NAME, "moreLink").get_attribute("href")
            
            rankings.append({
                'date': datetime.now().strftime('%Y-%m-%d'),
                'rank': int(rank.replace('#', '')),
                'team_id': team_id,
                'team_name': team_name,
                'points': int(points),
                'profile_link': profile_link
            })
        except Exception as e:
            print(f"Hiba: {e}")
            continue
    
    driver.quit()
    return pd.DataFrame(rankings)

# Futtasd hetente → time-series ranking data
rankings = scrape_team_rankings()
display(rankings)

In [ ]:
# Events

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

def scrape_major_events():
    driver = webdriver.Chrome()
    driver.get("https://www.hltv.org/events/archive?eventType=MAJOR")
    
    # Várunk, amíg betöltődnek az események hónapjai
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "events-month")))
    
    events_data = []
    months_divs = driver.find_elements(By.CLASS_NAME, "events-month")
    
    for month_div in months_divs:
        try:
            # Hónap
            month_name = month_div.find_element(By.CLASS_NAME, "standard-headline").text.strip()
            
            # Minden esemény az adott hónapban
            event_links = month_div.find_elements(By.CSS_SELECTOR, "a.small-event.standard-box")
            
            for event in event_links:
                try:
                    event_name = event.find_element(By.CSS_SELECTOR, ".event-col .text-ellipsis").text.strip()
                    team_count = event.find_elements(By.CSS_SELECTOR, ".table tr:first-child td.small-col")[0].text.strip()
                    prize = event.find_elements(By.CSS_SELECTOR, ".table tr:first-child td.prizePoolEllipsis")[0].get_attribute("title").strip()
                    link = event.get_attribute("href").strip()
                    event_id = link.split('/')[4]
                    
                    events_data.append({
                        "month": month_name,
                        "event_name": event_name,
                        "event_id": event_id,
                        "teams": team_count,
                        "prize": prize,
                        "link": link
                    })
                except Exception as e_event:
                    print(f"Esemény hiba: {e_event}")
                    continue
        except Exception as e_month:
            print(f"Hónap hiba: {e_month}")
            continue
    
    driver.quit()
    return pd.DataFrame(events_data)

# Futtatás
major_events = scrape_major_events()
display(major_events.sample(5))

In [ ]:
# Matches of event

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

def scrape_event_results(event_id):
    url = f"https://www.hltv.org/results?event={event_id}"
    driver = webdriver.Chrome()
    driver.get(url)
    
    wait = WebDriverWait(driver, 10)
    wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "results-sublist")))
    
    results_data = []
    sublists = driver.find_elements(By.CLASS_NAME, "results-sublist")
    
    for sublist in sublists:
        try:
            # Mérkőzés dátuma
            match_date = sublist.find_element(By.CLASS_NAME, "standard-headline").text.strip()
            
            # Az összes mérkőzés az adott dátumban
            matches = sublist.find_elements(By.CSS_SELECTOR, ".result-con a")
            
            for match in matches:
                try:
                    link = match.get_attribute("href").strip()
                    
                    team_home = match.find_element(By.CSS_SELECTOR, ".team1 .team").text.strip()
                    team_away = match.find_element(By.CSS_SELECTOR, ".team2 .team").text.strip()
                    
                    # Pontok a sorrend alapján
                    score_spans = match.find_elements(By.CSS_SELECTOR, ".result-score span")
                    score_home = int(score_spans[0].text.strip())
                    score_away = int(score_spans[1].text.strip())
                    
                    map_type = match.find_element(By.CSS_SELECTOR, ".map-and-stars .map-text").text.strip()
                    
                    rounds = int(map_type[-1]) if map_type[:2] == "bo" else 1

                    results_data.append({
                        "date": match_date,
                        "team_home": team_home,
                        "team_away": team_away,
                        "score_home": score_home,
                        "score_away": score_away,
                        "map": map_type,
                        "rounds": rounds,
                        "link": link
                    })
                except Exception as e_match:
                    print(f"Mérkőzés hiba: {e_match}")
                    continue
        except Exception as e_sublist:
            print(f"Dátum hiba: {e_sublist}")
            continue
    
    driver.quit()
    return pd.DataFrame(results_data)

# Példa
event_results = scrape_event_results(7902)
display(event_results.sample(5))


In [ ]:
# Match - Head-to-Head, Stats

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import numpy as np
import time

def scrape_head_to_head(match_url):
    driver = webdriver.Chrome()
    driver.get(match_url)
    wait = WebDriverWait(driver, 15)

    data = []

    try:
        # --- HEAD TO HEAD rész ---
        wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "head-to-head")))
        h2h_box = driver.find_element(By.CLASS_NAME, "head-to-head")
        container = h2h_box.find_element(By.CLASS_NAME, "standard-box")

        team1 = container.find_element(By.CSS_SELECTOR, ".team1 .teamName").text.strip()
        team2 = container.find_element(By.CSS_SELECTOR, ".team2 .teamName").text.strip()

        stats = container.find_elements(By.CSS_SELECTOR, ".flexbox-column.grow .bold")
        wins_team1 = int(stats[0].text.strip())
        overtimes = int(stats[1].text.strip())
        wins_team2 = int(stats[2].text.strip())
        total_non_ot = wins_team1 + wins_team2
        home_win_rate = wins_team1 / total_non_ot if total_non_ot > 0 else None

        # --- PLAYER STAT rész ---
        # Várunk, hogy a statisztika box betöltődjön
        wait.until(EC.presence_of_all_elements_located((By.CLASS_NAME, "stats-content")))
        time.sleep(2)

        stats_tables = driver.find_elements(By.CSS_SELECTOR, "table.totalstats")

        def parse_team_stats(table):
            """Parseolja a team táblát és visszaadja a statok listáját."""
            rows = table.find_elements(By.TAG_NAME, "tr")[1:]  # első sor a header
            team_name = table.find_element(By.CSS_SELECTOR, ".teamName.team").text.strip()

            ratings, adrs, swings = [], [], []

            for r in rows:
                try:
                    rating = float(r.find_element(By.CSS_SELECTOR, ".rating").text.strip())
                    adr = float(r.find_element(By.CSS_SELECTOR, ".adr").text.strip())
                    swing_text = r.find_element(By.CSS_SELECTOR, ".roundSwing").text.strip().replace('%','')
                    swing = float(swing_text.replace('+', '').replace(',', '.'))
                    ratings.append(rating)
                    adrs.append(adr)
                    swings.append(swing)
                except Exception:
                    continue

            return team_name, ratings, adrs, swings

        # Feltételezzük, hogy az első totalstats a home team (team1), a második az away (team2)
        home_team_name, home_ratings, home_adrs, home_swings = parse_team_stats(stats_tables[0])
        away_team_name, away_ratings, away_adrs, away_swings = parse_team_stats(stats_tables[1])

        data.append({
            "home_team": team1,
            "away_team": team2,
            "wins_home": wins_team1,
            "wins_away": wins_team2,
            "overtimes": overtimes,
            "total_non_overtime": total_non_ot,
            "home_win_rate": round(home_win_rate, 4) if home_win_rate is not None else None,
            "home_team_avg_rating": np.mean(home_ratings) if home_ratings else None,
            "home_team_std_rating": np.std(home_ratings) if home_ratings else None,
            "home_team_avg_ADR": np.mean(home_adrs) if home_adrs else None,
            "home_team_std_ADR": np.std(home_adrs) if home_adrs else None,
            "home_team_avg_Swing": np.mean(home_swings) if home_swings else None,
            "home_team_std_Swing": np.std(home_swings) if home_swings else None,
            "away_team_avg_rating": np.mean(away_ratings) if away_ratings else None,
            "away_team_std_rating": np.std(away_ratings) if away_ratings else None,
            "away_team_avg_ADR": np.mean(away_adrs) if away_adrs else None,
            "away_team_std_ADR": np.std(away_adrs) if away_adrs else None,
            "away_team_avg_Swing": np.mean(away_swings) if away_swings else None,
            "away_team_std_Swing": np.std(away_swings) if away_swings else None,
            "source_url": match_url
        })

    except Exception as e:
        print(f"Hiba a scraperben: {e}")

    driver.quit()
    return pd.DataFrame(data)


# Példa futtatás
url = "https://www.hltv.org/matches/2382612/virtuspro-vs-pain-blasttv-austin-major-2025"
df_h2h = scrape_head_to_head(url)
display(df_h2h)


In [ ]:
# Team history

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import datetime as dt
import time

def scrape_team_results_summary(team_id):
    url = f"https://www.hltv.org/results?team={team_id}"
    driver = webdriver.Chrome()
    driver.get(url)

    wait = WebDriverWait(driver, 15)
    wait.until(EC.presence_of_element_located((By.CLASS_NAME, "results-holder")))

    time.sleep(2)
    today = pd.Timestamp.now().normalize()

    matches = []

    try:
        results_holder = driver.find_element(By.CLASS_NAME, "results-holder")
        sublists = results_holder.find_elements(By.CLASS_NAME, "results-sublist")
        print(f"🔍 Talált results-sublist blokkok: {len(sublists)}")

        for sublist in sublists:
            # --- dátum fejléc ---
            headline_el = sublist.find_element(By.CLASS_NAME, "standard-headline")
            headline_text = headline_el.text.strip().replace("Results for", "").strip()
            date = pd.to_datetime(headline_text, errors='coerce')

            match_blocks = sublist.find_elements(By.CLASS_NAME, "result-con")
            for match in match_blocks:
                try:
                    a_tag = match.find_element(By.TAG_NAME, "a")
                    match_url = a_tag.get_attribute("href")
                    table = a_tag.find_element(By.TAG_NAME, "table")
                    tds = table.find_elements(By.TAG_NAME, "td")

                    team1_name = tds[0].find_element(By.CLASS_NAME, "team").text.strip()
                    team2_name = tds[2].find_element(By.CLASS_NAME, "team").text.strip()

                    # --- score ---
                    score_spans = tds[1].find_elements(By.TAG_NAME, "span")
                    score1 = int(score_spans[0].text.strip())
                    score2 = int(score_spans[1].text.strip())

                    # --- winner ---
                    team1_html = tds[0].get_attribute("innerHTML")
                    team2_html = tds[2].get_attribute("innerHTML")

                    if "team-won" in team1_html:
                        winner = team1_name
                    elif "team-won" in team2_html:
                        winner = team2_name
                    else:
                        winner = None

                    # --- target team name detektálás ---
                    is_target_team1 = str(team_id) in match_url and team1_name != ""  # fallback
                    is_target_team2 = str(team_id) in match_url and team2_name != ""  # fallback

                    # egyszerűbb: bármelyik oldalon is van a csapat, figyeljük, hogy nyert-e
                    target_team_name = team1_name if (score1 > score2) or ("team-won" in team1_html) else team2_name
                    result = "win" if score1 > score2 else "loss"

                    matches.append({
                        "date": date,
                        "team1": team1_name,
                        "team2": team2_name,
                        "score1": score1,
                        "score2": score2,
                        "winner": winner,
                        "result": result
                    })

                except Exception as e:
                    print(f"⚠️ Hiba meccs feldolgozásnál: {e}")
                    continue

        driver.quit()

        # --- DataFrame feldolgozás ---
        df = pd.DataFrame(matches)
        if df.empty:
            print("⚠️ Nincs adat!")
            return pd.DataFrame()

        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.dropna(subset=["date"]).sort_values("date", ascending=False)
        df["days_ago"] = (today - df["date"]).dt.days

        # --- winrate számítás ---
        def winrate(df):
            if len(df) == 0:
                return None
            return round((df["result"] == "win").sum() / len(df), 4)

        last_30d = df[df["days_ago"] <= 30]
        last_90d = df[df["days_ago"] <= 90]
        last_7d = df[df["days_ago"] <= 7]

        last_30d_winrate = winrate(last_30d)
        last_90d_winrate = winrate(last_90d)
        matches_last_7d = len(last_7d)

        last_match_date = df["date"].max()
        days_since_last_match = int((today - last_match_date).days) if pd.notna(last_match_date) else None

        # --- current streak ---
        current_streak = 0
        if len(df) > 0:
            last_results = df["result"].tolist()
            last_outcome = last_results[0]
            for r in last_results:
                if r == last_outcome:
                    current_streak += 1
                else:
                    break
            if last_outcome == "loss":
                current_streak *= -1  # negatív streak veszteség esetén

        df_summary = pd.DataFrame([{
            "team_id": team_id,
            "scrape_date": today,
            "last_30d_winrate": last_30d_winrate,
            "last_90d_winrate": last_90d_winrate,
            "matches_last_7d": matches_last_7d,
            "days_since_last_match": days_since_last_match,
            "current_streak": current_streak,
            "total_matches_found": len(df),
            "source_url": url
        }])

        print(f"✅ Összesen {len(df)} meccs feldolgozva a csapatnál.")
        return df_summary

    except Exception as e:
        print(f"❌ Hiba a scraperben: {e}")
        driver.quit()
        return pd.DataFrame()


# --- Példa futtatás ---
team_id = 5378  # Virtus.pro
df_team_summary = scrape_team_results_summary(team_id)
display(df_team_summary)
